In [1]:
import pyperclip

Heading

Position
Segment
Name
Max use

R4 Loop MandatoryRepeat 20



In [17]:
raw = '''
010
ST
Transaction Set HeaderMandatory->
Max 1
To indicate the start of a transaction set and to assign a control number

020
B4
Beginning Segment for Inquiry or ReplyMandatory->
Max 1
To transmit identifying numbers, dates, and other basic data relating to the transaction set

030
N9
Reference IdentificationOptional->
Max 30
To transmit identifying information as specified by the Reference Identification Qualifier

040
Q2
Status Details (Ocean)Optional->
Max 1
To transmit identifying information relative to identification of vessel, transportation dates, lading quantity, weight, and cube

050
SG
Shipment StatusOptional->
Max 15
To convey the status of a shipment


060
R4
Port or TerminalMandatory->
Max 1
Contractual or operational port or point relevant to the movement of the cargo

070
DTM
Date/Time ReferenceOptional->
Max 15
To specify pertinent dates and times

080
V9
Event DetailOptional->
Max 10
To specify information about a specific event

090
SE
Transaction Set TrailerMandatory->
Max 1
To indicate the end of the transaction set and provide the count of the transmitted segments (including the beginning (ST) and ending (SE) segments)

'''

In [103]:
data: list = raw.strip().split('\n')

positions = [_ for _ in data if _.isdigit()]
segments = [_ for _ in data if _ and not _.isdigit() and len(_) <4]
req = ['Mandatory' if 'Mandatory' in _ else 'Optional' if 'Optional' in _ else _ for _ in data if '->' in _]
seg_name = [_.split('Mandatory')[0].split('Optional')[0] for _ in data if '->' in _]
max_len = [_.split()[-1] for _ in data if 'Max' in _]
description = [_ for _ in data if len(_) > 10 and '->' not in _]

In [209]:
import pandas as pd

tmp = {
    'Posn': positions,
    'SegAbbrev': segments,
    'SegName': seg_name,
    'Req': req,
    'Description': description,
    'Max': max_len,
}

df = pd.DataFrame(tmp)

df.insert(0, 'Owner', 'X12')
df.insert(1, 'Release', '4010')
df.insert(2, 'EDI', '315')
df.insert(3, 'Level', 'Heading')
df.insert(4, 'Loop', '')
df.loc[df.Posn.isin(['060', '070']), 'Loop'] = 'R4'


df.insert(7, 'Elements', '')

df.loc[df.SegAbbrev == 'ST', 'Elements'] = 2
df.loc[df.SegAbbrev == 'B4', 'Elements'] = 13
df.loc[df.SegAbbrev == 'N9', 'Elements'] = 13
df.loc[df.SegAbbrev == 'Q2', 'Elements'] = 16
df.loc[df.SegAbbrev == 'SG', 'Elements'] = 6
df.loc[df.SegAbbrev == 'R4', 'Elements'] = 8
df.loc[df.SegAbbrev == 'DTM', 'Elements'] = 6
df.loc[df.SegAbbrev == 'V9', 'Elements'] = 20
df.loc[df.SegAbbrev == 'SE', 'Elements'] = 2



df


,Owner,Release,EDI,Level,Loop,Posn,SegAbbrev,Elements,SegName,Req,Description,Max
0,X12,4010,315,Heading,,010,ST,2,Transaction Set Header,Mandatory,To indicate the start of a transaction set and...,1
1,X12,4010,315,Heading,,020,B4,13,Beginning Segment for Inquiry or Reply,Mandatory,"To transmit identifying numbers, dates, and ot...",1
2,X12,4010,315,Heading,,030,N9,13,Reference Identification,Optional,To transmit identifying information as specifi...,30
3,X12,4010,315,Heading,,040,Q2,16,Status Details (Ocean),Optional,To transmit identifying information relative t...,1
4,X12,4010,315,Heading,,050,SG,6,Shipment Status,Optional,To convey the status of a shipment,15
5,X12,4010,315,Heading,R4,060,R4,8,Port or Terminal,Mandatory,Contractual or operational port or point relev...,1
6,X12,4010,315,Heading,R4,070,DTM,6,Date/Time Reference,Optional,To specify pertinent dates and times,15
7,X12,4010,315,Heading,,080,V9,20,Event Detail,Optional,To specify information about a specific event,10
8,X12,4010,315,Heading,,090,SE,2,Transaction Set Trailer,Mandatory,To indicate the end of the transaction set and...,1


In [333]:
isa = '''
ISA-01
I01
Authorization Information Qualifier
Identifier (ID)
Mandatory
2
2
-
Code to identify the type of information in the Authorization Information
Codes (7)

|||

ISA-02
I02
Authorization Information
String (AN)
Mandatory
10
10
-
Information used for additional identification or authorization of the interchange sender or the data in the interchange; the type of information is set by the Authorization Information Qualifier (I01)


|||


ISA-03
I03
Security Information Qualifier
Identifier (ID)
Mandatory
2
2
-
Code to identify the type of information in the Security Information
Codes (2)


|||


ISA-04
I04
Security Information
String (AN)
Mandatory
10
10
-
This is used for identifying the security information about the interchange sender or the data in the interchange; the type of information is set by the Security Information Qualifier (I03)


|||


ISA-05
I05
Interchange ID Qualifier
Identifier (ID)
Mandatory
2
2
-
Qualifier to designate the system/method of code structure used to designate the sender or receiver ID element being qualified
Codes (38)


|||


ISA-06
I06
Interchange Sender ID
String (AN)
Mandatory
15
15
-
Identification code published by the sender for other parties to use as the receiver ID to route data to them; the sender always codes this value in the sender ID element


|||


ISA-07
I05
Interchange ID Qualifier
Identifier (ID)
Mandatory
2
2
-
Qualifier to designate the system/method of code structure used to designate the sender or receiver ID element being qualified
Codes (38)

|||


ISA-08
I07
Interchange Receiver ID
String (AN)
Mandatory
15
15
-
Identification code published by the receiver of the data; When sending, it is used by the sender as their sending ID, thus other parties sending to them will use this as a receiving ID to route data to them

|||


ISA-09
I08
Interchange Date
Date (DT)
Mandatory
6
6
-
Date of the interchange

|||


ISA-10
I09
Interchange Time
Time (TM)
Mandatory
4
4
-
Time of the interchange

|||


ISA-11
I10
Interchange Control Standards Identifier
Identifier (ID)
Mandatory
1
1
-
Code to identify the agency responsible for the control standard used by the message that is enclosed by the interchange header and trailer
Codes (1)

|||


ISA-12
I11
Interchange Control Version Number
Identifier (ID)
Mandatory
5
5
-
This version number covers the interchange control segments
Codes (14)

|||

ISA-13
I12
Interchange Control Number
Numeric (N0)
Mandatory
9
9
-
A control number assigned by the interchange sender

|||


ISA-14
I13
Acknowledgment Requested
Identifier (ID)
Mandatory
1
1
-
Code sent by the sender to request an interchange acknowledgment (TA1)
Codes (2)

|||


ISA-15
I14
Usage Indicator
Identifier (ID)
Mandatory
1
1
-
Code to indicate whether data enclosed by this interchange envelope is test, production or information
Codes (3)

|||


ISA-16
I15
Component Element Separator
Mandatory
1
1
-
Type is not applicable; the component element separator is a delimiter and not a data element; this field provides the delimiter used to separate component data elements within a composite data structure; this value must be different than the data element separator and the segment terminator


'''


data: list = [_.strip().split('\n') for _ in isa.strip().split('|||')]

data

[['ISA-01',
  'I01',
  'Authorization Information Qualifier',
  'Identifier (ID)',
  'Mandatory',
  '2',
  '2',
  '-',
  'Code to identify the type of information in the Authorization Information',
  'Codes (7)'],
 ['ISA-02',
  'I02',
  'Authorization Information',
  'String (AN)',
  'Mandatory',
  '10',
  '10',
  '-',
  'Information used for additional identification or authorization of the interchange sender or the data in the interchange; the type of information is set by the Authorization Information Qualifier (I01)'],
 ['ISA-03',
  'I03',
  'Security Information Qualifier',
  'Identifier (ID)',
  'Mandatory',
  '2',
  '2',
  '-',
  'Code to identify the type of information in the Security Information',
  'Codes (2)'],
 ['ISA-04',
  'I04',
  'Security Information',
  'String (AN)',
  'Mandatory',
  '10',
  '10',
  '-',
  'This is used for identifying the security information about the interchange sender or the data in the interchange; the type of information is set by the Security 

In [359]:
[_[6] if _[3] != 'Mandatory' else _[5] for _ in data]

['2',
 '10',
 '2',
 '10',
 '2',
 '15',
 '2',
 '15',
 '6',
 '4',
 '1',
 '5',
 '9',
 '1',
 '1',
 '1']

In [473]:
elem_abbrev = [_[0] for _ in data]
data_element_id = [_[1] for _ in data]
element_name = [_[2] for _ in data]
data_type = [_[3] if _[3] != 'Mandatory' else '' for _ in data]
req = [_[4] if _[3] != 'Mandatory' else _[3] for _ in data]
min_len = [_[5] if _[3] != 'Mandatory' else _[4] for _ in data]
max_len = [_[6] if _[3] != 'Mandatory' else _[5] for _ in data]
description = [_[8] if _[3] != 'Mandatory' else _[7] for _ in data]
codes = [_[9].split('(')[-1].split(')')[0] if len(_) > 9 else '' for _ in data]  



import pandas as pd

tmp = {
    'ElemAbbrev': elem_abbrev,
    'DE': data_element_id,
    'ElemName': element_name,
    'Type': data_type,
    'Codes': codes,
    'Req': req,
    'Min': min_len,
    'Max': max_len,
    'Description': description,
}

df = pd.DataFrame(tmp)

df.insert(0, 'Owner', 'X12')
df.insert(1, 'Release', '4010')
df.insert(2, 'Level', 'Interchange')

status = ['Drop (not used)', 'Drop (not used)', 'Drop (not used)', 'Drop (not used)', 
          'Add meeaning of code', 'Who sent this', 'Add meeaning of code', 'Who Sent to', 
          'Drop (Add DTS)', 'Drop (Add DTS)', 'Drop (always U)', 'Convert to correct format', 
          'Unique ID', 'YES/NO', 'P/T/I', 'Drop (ony used to get delimiter)']

df.insert(3, 'Status', status)

df



,Owner,Release,Level,Status,ElemAbbrev,DE,ElemName,Type,Codes,Req,Min,Max,Description
0,X12,4010,Interchange,Drop (not used),ISA-01,I01,Authorization Information Qualifier,Identifier (ID),7,Mandatory,2,2,Code to identify the type of information in th...
1,X12,4010,Interchange,Drop (not used),ISA-02,I02,Authorization Information,String (AN),,Mandatory,10,10,Information used for additional identification...
2,X12,4010,Interchange,Drop (not used),ISA-03,I03,Security Information Qualifier,Identifier (ID),2,Mandatory,2,2,Code to identify the type of information in th...
3,X12,4010,Interchange,Drop (not used),ISA-04,I04,Security Information,String (AN),,Mandatory,10,10,This is used for identifying the security info...
4,X12,4010,Interchange,Add meeaning of code,ISA-05,I05,Interchange ID Qualifier,Identifier (ID),38,Mandatory,2,2,Qualifier to designate the system/method of co...
5,X12,4010,Interchange,Who sent this,ISA-06,I06,Interchange Sender ID,String (AN),,Mandatory,15,15,Identification code published by the sender fo...
6,X12,4010,Interchange,Add meeaning of code,ISA-07,I05,Interchange ID Qualifier,Identifier (ID),38,Mandatory,2,2,Qualifier to designate the system/method of co...
7,X12,4010,Interchange,Who Sent to,ISA-08,I07,Interchange Receiver ID,String (AN),,Mandatory,15,15,Identification code published by the receiver ...
8,X12,4010,Interchange,Drop (Add DTS),ISA-09,I08,Interchange Date,Date (DT),,Mandatory,6,6,Date of the interchange
9,X12,4010,Interchange,Drop (Add DTS),ISA-10,I09,Interchange Time,Time (TM),,Mandatory,4,4,Time of the interchange


In [497]:
df1 = df[['ElemName', 'Status', 'ElemAbbrev']].T
df1.insert(0, 'Seg', 'ISA')
df1 = df1.reset_index(drop=True)
df1

,Seg,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,ISA,Authorization Information Qualifier,Authorization Information,Security Information Qualifier,Security Information,Interchange ID Qualifier,Interchange Sender ID,Interchange ID Qualifier,Interchange Receiver ID,Interchange Date,Interchange Time,Interchange Control Standards Identifier,Interchange Control Version Number,Interchange Control Number,Acknowledgment Requested,Usage Indicator,Component Element Separator
1,ISA,Drop (not used),Drop (not used),Drop (not used),Drop (not used),Add meeaning of code,Who sent this,Add meeaning of code,Who Sent to,Drop (Add DTS),Drop (Add DTS),Drop (always U),Convert to correct format,Unique ID,YES/NO,P/T/I,Drop (ony used to get delimiter)
2,ISA,ISA-01,ISA-02,ISA-03,ISA-04,ISA-05,ISA-06,ISA-07,ISA-08,ISA-09,ISA-10,ISA-11,ISA-12,ISA-13,ISA-14,ISA-15,ISA-16


In [421]:
print(['Drop (not used)',
 'Drop (not used)',
 'Drop (not used)',
 'Drop (not used)',
 'Interchange ID Qualifier (add meeaning of code)',
 'Interchange Sender ID',
 'Interchange ID Qualifier (add meaning of code)',
 'Interchange Receiver ID',
 'Interchange Date (convert to DTS)', 
 'Interchange Time(convert to DTS)',
 'Drop (always U)',
 'Interchange Control Version Number (Release)',
 'Interchange Control Number (Unique ID)',
 'Acknowledgment Requested (YES/NO',
 'Usage Indicator (PTI)',
    'Drop (ony used to get delimiter)',
 

])

['Drop (not used)', 'Drop (not used)', 'Drop (not used)', 'Drop (not used)', 'Interchange ID Qualifier (add meeaning of code)', 'Interchange Sender ID', 'Interchange ID Qualifier (add meaning of code)', 'Interchange Receiver ID', 'Interchange Date (convert to DTS)', 'Interchange Time(convert to DTS)', 'Drop (always U)', 'Interchange Control Version Number (Release)', 'Interchange Control Number (Unique ID)', 'Acknowledgment Requested (YES/NO', 'Usage Indicator (PTI)', 'Drop (ony used to get delimiter)']
